In [1]:
from pathlib import Path
import subprocess
import sys

from instance_reader import read_instance
from initial_solution import greedy_initial_solution
from alns import (
    run_alns,
    SAParams,
    ALNSParams,
    PenaltyParams,
    RandomRemoval,
    WorstDensityRemoval,
    GreedyBestInsertionRepair,
    Regret2InsertionRepair,
)

In [2]:
instance_path = Path("dataset/skillvrp_n400_v10_s8_k2.0_10.txt")
checker_path = Path("checker.py")

solution_dir = Path("solutions")
solution_dir.mkdir(parents=True, exist_ok=True)

solution_path = solution_dir / "solution.txt"

In [3]:
inst = read_instance(str(instance_path))

In [4]:
start_solution = greedy_initial_solution(
    inst,
    strategy="multi_start",
    verbose=True,
)

print("Initial objective:", start_solution.objective)

Greedy strategy: hybrid
  Objective: 9750
  Served customers: 128/400
Greedy strategy: profit
  Objective: 10426
  Served customers: 122/400
Greedy strategy: density
  Objective: 10255
  Served customers: 130/400
Greedy strategy: fewest_vehicles
  Objective: 10035
  Served customers: 122/400

Selected greedy strategy: profit
Best objective: 10426
Served customers: 122/400
Initial objective: 10426


In [5]:
sa = SAParams(
    initial_temperature=100.0,
    min_temperature=0.001,
    cooling_rate=0.9995,
    reheat_factor=0.75,
)

penalties = PenaltyParams(
    time_window_penalty=20.0,
    shift_penalty=20.0,
    skill_penalty=10000.0,
)

params = ALNSParams(
    random_seed=1,
    segment_length=100,
    no_accept_limit=500,
    reaction_factor=0.20,
    min_operator_weight=0.05,
    score_global_best=25.0,
    score_better_current=10.0,
    score_accepted=3.0,
    score_rejected=0.0,
    time_cost_alpha=0.5,
    time_scale_seconds=0.01,
    verbose=True,
)


destroy_operators = [
    RandomRemoval(
        fraction=0.15,
        min_remove=1,
        max_remove=50,
        initial_weight=1.0,
    ),
    WorstDensityRemoval(
        fraction=0.15,
        min_remove=1,
        max_remove=50,
        noise=0.05,
        initial_weight=1.0,
    ),
]

repair_operators = [
    GreedyBestInsertionRepair(
        extra_unserved_limit=100,
        max_insertions=None,
        min_delta_score=0.0,
        initial_weight=1.0,
    ),
    Regret2InsertionRepair(
        extra_unserved_limit=100,
        max_insertions=None,
        min_delta_score=0.0,
        initial_weight=1.0,
    ),
]

In [6]:
result = run_alns(
    runtime=30.0,
    inst=inst,
    start_solution=start_solution,
    sa=sa,
    params=params,
    penalties=penalties,
    destroy_operators=destroy_operators,
    repair_operators=repair_operators,
)

print()
print("ALNS finished")
print("Best objective:", result.best_evaluation.profit)
print("Best penalty:", result.best_evaluation.penalty)
print("Best feasible:", result.best_evaluation.feasible)
print("Iterations:", result.iterations)
print("Restarts:", result.restarts)


ALNS finished
Best objective: 11643
Best penalty: 0.0
Best feasible: True
Iterations: 48
Restarts: 0


In [7]:
result.best_solution.write_for_checker(str(solution_path))

In [8]:
checker_result = subprocess.run(
    [
        sys.executable,
        str(checker_path),
        str(instance_path),
        str(solution_path),
    ],
    capture_output=True,
    text=True,
)

print()
print("Checker stdout:")
print(checker_result.stdout)

if checker_result.stderr:
    print("Checker stderr:")
    print(checker_result.stderr)

print("Checker return code:", checker_result.returncode)


Checker stdout:
OK: feasible, profit = 11643, visited 142/400 customers

Checker return code: 0
